In [ ]:
import pandas as pd
import os

# 각 파일의 경로 저장
file1 = './train/admissions_train.csv'  # 얘를 기준으로
file2 = './train/chartevents_train.csv'
file3 = './train/icustays_train.csv'
file4 = './train/ingredientevents_train.csv'
file5 = './train/inputevents_train.csv'
file6 = './train/outputevents_train.csv'
file7 = './train/patients_train.csv'  # 이 파일에는 hadm_id에 대한 정보가 없음
file8 = './train/procedureevents_train.csv'

# 1. admissions_train_F.csv 읽기
admissions_data = pd.read_csv(file1)

# 2. admissions_train_F.csv에서 hadm_id 리스트 추출 및 중복 확인
hadm_id_list = admissions_data['hadm_id'].unique()  # 중복 제거된 고유 hadm_id 리스트 생성
print(f"admissions_train_F.csv에서 추출된 고유 hadm_id 개수: {len(hadm_id_list)}")

# 3. 각 파일 읽고 hadm_id 기준으로 필터링
file_paths = [file2, file3, file4, file5, file6, file7, file8]
filtered_data = {}

for idx, file_path in enumerate(file_paths, start=2):
    # 파일 읽기
    current_data = pd.read_csv(file_path)

    # hadm_id가 있는 경우에만 필터링
    if 'hadm_id' in current_data.columns:
        # 필터링 전 데이터 크기
        original_shape = current_data.shape

        # hadm_id 필터링
        filtered_data[file_path] = current_data[current_data['hadm_id'].isin(hadm_id_list)]

        # 필터링 후 데이터 크기
        filtered_shape = filtered_data[file_path].shape

        # 차이나는 데이터 개수
        difference_count = original_shape[0] - filtered_shape[0]

        # 필터링 전, 후 데이터 크기 및 차이 출력
        print(f"{file_path}: 필터링 전 데이터 크기: {original_shape}, 필터링 후 데이터 크기: {filtered_shape}, 차이: {difference_count}")
    else:
        # hadm_id가 없는 경우 필터링 없이 저장
        filtered_data[file_path] = current_data
        print(f"{file_path}는 hadm_id가 없으므로 필터링하지 않음. 데이터 크기: {current_data.shape}")

# 4. 각 필터링된 데이터를 저장 (train_F 폴더에 저장)
output_folder = './train_F/'  # 저장할 폴더 경로

# 폴더가 존재하지 않으면 생성
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# admissions_train.csv 복사하여 저장
admissions_output_path = os.path.join(output_folder, 'admissions_train_F.csv')
admissions_data.to_csv(admissions_output_path, index=False)
print(f"admissions_train.csv를 복사하여 저장: {admissions_output_path}")

# 다른 파일 저장
for file_path, data in filtered_data.items():
    # 파일 이름에서 폴더 경로 제거 후, train_F 폴더에 저장 경로 생성
    filename = os.path.basename(file_path).replace('.csv', '_F.csv')  # 파일 이름 변경
    output_path = os.path.join(output_folder, filename)  # train_F 폴더에 저장 경로 설정

    # 필터링된 데이터 저장
    data.to_csv(output_path, index=False)
    print(f"필터링된 데이터 저장: {output_path}")


In [ ]:
import pandas as pd

# 파일 경로 정의
admissions_file = "./train_F/admissions_train_F.csv"

# admissions_train_F.csv 데이터 로드
print("admissions_train_F.csv 데이터 로드 중...")
admissions_data = pd.read_csv(admissions_file)

# hospital_expire_flag 열 확인
if "hospital_expire_flag" not in admissions_data.columns:
    raise KeyError("admissions_train_F.csv 파일에 'hospital_expire_flag' 열이 없습니다.")

# hadm_id와 hospital_expire_flag 매칭 확인
missing_hadm_ids = admissions_data[admissions_data["hospital_expire_flag"].isna()]

# 결과 출력
missing_count = len(missing_hadm_ids)
print(f"hospital_expire_flag가 없는 hadm_id 개수: {missing_count}")

# 누락된 hadm_id 확인
if missing_count > 0:
    print(f"누락된 hadm_id 목록:\n{missing_hadm_ids['hadm_id'].tolist()}")


In [ ]:
import pandas as pd
import time
from tqdm import tqdm

# 파일 경로 정의
file_paths = {
    "admissions_train_F.csv": {
        "time_columns": [("admittime", "admittime"), ("dischtime", "dischtime")],
    },
    "chartevents_train_F.csv": {
        "time_columns": [("charttime", "charttime")],
    },
    "icustays_train_F.csv": {
        "time_columns": [("intime", "icuintime"), ("outtime", "icuouttime")],
    },
    "ingredientevents_train_F.csv": {
        "time_columns": [("starttime", "starttime")],
    },
    "inputevents_train_F.csv": {
        "time_columns": [("starttime", "starttime")],
    },
    "outputevents_train_F.csv": {
        "time_columns": [("charttime", "charttime")],
    },
    "procedureevents_train_F.csv": {
        "time_columns": [("starttime", "starttime")],
    },
}

# 결과 저장 파일
output_file = "./train_F/timeline.csv"

# admissions_train_F.csv에서 hospital_expire_flag 정보 읽기
admissions_file = "./train_F/admissions_train_F.csv"
print("admissions_train_F.csv 데이터 로드 중...")
admissions_data = pd.read_csv(admissions_file)

# hospital_expire_flag 존재 여부 확인
if "hospital_expire_flag" not in admissions_data.columns:
    raise KeyError("admissions_train_F.csv 파일에 'hospital_expire_flag' 열이 없습니다.")

# 필요한 열만 추출
admissions_data = admissions_data[["hadm_id", "hospital_expire_flag"]]

# 결과 저장용 리스트
result_data = []

# 전체 작업 시작 시간
start_time = time.time()

# 전체 작업 진행
total_files = len(file_paths)
for i, (file_name, params) in enumerate(file_paths.items(), start=1):
    print(f"작업 시작: {file_name} ({i}/{total_files})")
    file_start_time = time.time()
    
    # 파일 경로 및 데이터 크기 확인
    file_path = f"./train_F/{file_name}"
    chunk_size = 10000  # 데이터 청크 크기

    # 데이터 청크 단위로 처리
    for chunk in tqdm(pd.read_csv(file_path, chunksize=chunk_size), desc=f"{file_name} 처리 중"):
        for time_col, event_type_name in params["time_columns"]:
            if time_col in chunk.columns:
                # event_time과 event_type 생성
                chunk["event_time"] = pd.to_datetime(chunk[time_col], errors="coerce")
                chunk["event_type"] = event_type_name
                
                # itemid 열 생성 (없으면 0)
                chunk["itemid"] = chunk["itemid"] if "itemid" in chunk.columns else 0
                
                # value 열 생성 (value > amount > 0 순으로 채움)
                if "value" in chunk.columns and "amount" in chunk.columns:
                    chunk["value"] = chunk["value"].fillna(chunk["amount"]).fillna(0)
                elif "value" in chunk.columns:
                    chunk["value"] = chunk["value"].fillna(0)
                elif "amount" in chunk.columns:
                    chunk["value"] = chunk["amount"].fillna(0)
                else:
                    chunk["value"] = 0
                
                # 필요한 컬럼 선택
                selected_columns = ["hadm_id", "event_time", "event_type", "itemid", "value"]
                result_data.append(chunk[selected_columns])
    
    # 파일 처리 완료
    print(f"{file_name} 작업 완료. 처리 시간: {time.time() - file_start_time:.2f}초.")

# 모든 청크를 하나의 DataFrame으로 합치기
final_result = pd.concat(result_data, ignore_index=True)

# hospital_expire_flag를 hadm_id 기준으로 병합
print("hospital_expire_flag 열 추가 중...")
final_result = final_result.merge(admissions_data, on="hadm_id", how="left")

# hospital_expire_flag를 첫 번째 열로 이동
columns = ["hospital_expire_flag"] + [col for col in final_result.columns if col != "hospital_expire_flag"]
final_result = final_result[columns]

# 결과를 CSV 파일로 저장
final_result.to_csv(output_file, index=False)

# 전체 작업 완료
total_elapsed_time = time.time() - start_time
print(f"모든 작업 완료. 총 처리 시간: {total_elapsed_time:.2f}초. 결과 저장: {output_file}")


In [ ]:
import pandas as pd

# 파일 경로 정의
fixed_file = "./train_F/timeline.csv"
sorted_file = "./train_F/timeline_sorted.csv"

# 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(fixed_file)

# datetime 형식으로 변환
print("event_time 열을 datetime 형식으로 변환 중...")
df["event_time"] = pd.to_datetime(df["event_time"])

# 데이터 정렬
print("데이터 정렬 중...")
df_sorted = df.sort_values(by=["hadm_id", "event_time"]).reset_index(drop=True)

# 정렬된 데이터 저장
print("정렬된 데이터를 저장 중...")
df_sorted.to_csv(sorted_file, index=False)

print(f"정렬 완료. 정렬된 파일 저장 경로: {sorted_file}")


In [ ]:
import pandas as pd

# 파일 경로 정의
file1 = "./train_F/timeline_sorted.csv"
file2 = "./train_F/timeline_updated.csv"

# 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(file1)

# datetime 형식으로 변환
print("event_time 열을 datetime 형식으로 변환 중...")
df["event_time"] = pd.to_datetime(df["event_time"])

# 경과 시간 계산
print("경과 시간 계산 중...")
df["elapsed_time"] = 0.0  # 새로운 열 추가
for hadm_id, group in df.groupby("hadm_id"):
    # admittime 기준 시간
    admittime = group.loc[group["event_type"] == "admittime", "event_time"].min()
    
    # 경과 시간을 시간 단위로 계산
    df.loc[group.index, "elapsed_time"] = (
        (group["event_time"] - admittime).dt.total_seconds() / 3600
    ).round(2)

# elapsed_time이 120 이상인 hadm_id 확인
print("elapsed_time이 120 이상인 경우의 hospital_expire_flag를 0으로 설정 중...")
hadm_ids_to_update = df.loc[df["elapsed_time"] >= 120, "hadm_id"].unique()

# hospital_expire_flag 업데이트 전 개수 저장
original_flag_count = df.loc[df["hadm_id"].isin(hadm_ids_to_update), "hospital_expire_flag"].sum()

# hospital_expire_flag를 0으로 변경
df.loc[df["hadm_id"].isin(hadm_ids_to_update), "hospital_expire_flag"] = 0

# hospital_expire_flag 변경된 개수 계산
updated_flag_count = original_flag_count - df.loc[df["hadm_id"].isin(hadm_ids_to_update), "hospital_expire_flag"].sum()
print(f"hospital_expire_flag를 0으로 변경한 hadm_id 개수: {len(hadm_ids_to_update)}")
print(f"hospital_expire_flag를 0으로 변경한 총 값 개수: {updated_flag_count}")

# 열 순서 변경: elapsed_time을 event_time 다음으로 이동
cols = list(df.columns)
cols.insert(cols.index("event_time") + 1, cols.pop(cols.index("elapsed_time")))
df = df[cols]

# event_time 열 제거
print("event_time 열 제거 중...")
df = df.drop(columns=["event_time"])

# 결과 저장
print("정제된 데이터를 저장 중...")
df.to_csv(file2, index=False)

print(f"경과 시간 계산 및 정제 완료. 업데이트된 파일 저장 경로: {file2}")


In [ ]:
import pandas as pd

# 업데이트된 데이터 로드
file2 = "./train_F/timeline_updated.csv"
print("업데이트된 데이터 로드 중...")
df = pd.read_csv(file2)

# hadm_id별 hospital_expire_flag가 유일한지 확인
print("hadm_id별 hospital_expire_flag 확인 중...")
conflicting_hadm_ids = df.groupby("hadm_id")["hospital_expire_flag"].nunique()
conflicting_hadm_ids = conflicting_hadm_ids[conflicting_hadm_ids > 1]

# 결과 출력
if not conflicting_hadm_ids.empty:
    print("다음 hadm_id는 서로 다른 hospital_expire_flag 값을 가지고 있습니다:")
    print(conflicting_hadm_ids.index.tolist())
else:
    print("모든 hadm_id는 유일한 hospital_expire_flag 값을 가지고 있습니다.")


In [ ]:
import pandas as pd

# 파일 경로 정의
file2 = "./train_F/timeline_updated.csv"
file3 = "./train_F/timeline2.csv"

# 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(file2)

# 초기 데이터 개수 저장
initial_data_count = len(df)

# 조건에 맞는 데이터 제거
print("elapsed_time 조건에 맞는 데이터 제거 중...")
filtered_df = df[(df["elapsed_time"] >= 0) & (df["elapsed_time"] < 120)]

# 제거된 데이터 개수 계산
removed_data_count = initial_data_count - len(filtered_df)

# 데이터 수 검증
print("데이터 수 검증 중...")
if initial_data_count - removed_data_count == len(filtered_df):
    print(f"검증 완료: 처음 데이터 수({initial_data_count}) - 제거된 데이터 수({removed_data_count}) = 결과 데이터 수({len(filtered_df)})")
else:
    print(f"검증 실패: 계산이 맞지 않습니다. 결과 데이터 수({len(filtered_df)})가 예상과 다릅니다.")

# 결과 저장
print("필터링된 데이터를 저장 중...")
filtered_df.to_csv(file3, index=False)

print(f"필터링 완료. 업데이트된 파일 저장 경로: {file3}")


In [ ]:
import pandas as pd

# 파일 경로 정의
input_file = "./train_F/timeline2.csv"

# 데이터 로드
print("파일 로드 중...")
df = pd.read_csv(input_file)

# 고유한 hadm_id에 대해 hospital_expire_flag 통계 계산
print("hospital_expire_flag 통계 계산 중...")
unique_hadm_flag = df[["hadm_id", "hospital_expire_flag"]].drop_duplicates()

# hospital_expire_flag가 1인 경우와 0인 경우 개수
flag_1_count = unique_hadm_flag[unique_hadm_flag["hospital_expire_flag"] == 1].shape[0]
flag_0_count = unique_hadm_flag[unique_hadm_flag["hospital_expire_flag"] == 0].shape[0]

# 결과 출력
print(f"hospital_expire_flag가 1인 고유 hadm_id 수: {flag_1_count}")
print(f"hospital_expire_flag가 0인 고유 hadm_id 수: {flag_0_count}")


In [ ]:
import pandas as pd

# 파일 경로 정의
input_file = "./train_F/timeline2.csv"

# 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(input_file)

# hadm_id별 데이터 개수 계산
print("hadm_id별 데이터 개수 분석 중...")
hadm_id_counts = df["hadm_id"].value_counts()

# 최대 및 최소 값 계산
max_count = hadm_id_counts.max()
min_count = hadm_id_counts.min()

# 최대 및 최소 hadm_id 확인
max_hadm_id = hadm_id_counts.idxmax()
min_hadm_id = hadm_id_counts.idxmin()

# 결과 출력
print(f"hadm_id별 데이터 개수 분석 결과:")
print(f"최대 데이터 개수: {max_count} (hadm_id: {max_hadm_id})")
print(f"최소 데이터 개수: {min_count} (hadm_id: {min_hadm_id})")

# 추가 통계 정보
print(f"전체 hadm_id 수: {hadm_id_counts.count()}")
print(f"데이터 개수의 평균: {hadm_id_counts.mean():.2f}")
print(f"데이터 개수의 중앙값: {hadm_id_counts.median():.2f}")


In [ ]:
import pandas as pd

# 파일 경로 정의
input_file = "./train_F/timeline2.csv"

# 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(input_file)

# hadm_id별 유니크한 elapsed_time 개수 계산
print("hadm_id별 유니크한 elapsed_time 개수 계산 중...")
unique_elapsed_time_counts = df.groupby("hadm_id")["elapsed_time"].nunique()

# 최대 및 최소 값 계산
max_count = unique_elapsed_time_counts.max()
min_count = unique_elapsed_time_counts.min()

# 최대 및 최소 hadm_id 확인
max_hadm_id = unique_elapsed_time_counts.idxmax()
min_hadm_id = unique_elapsed_time_counts.idxmin()

# 결과 출력
print(f"hadm_id별 유니크한 elapsed_time 개수 분석 결과:")
print(f"최대 유니크 elapsed_time 개수: {max_count} (hadm_id: {max_hadm_id})")
print(f"최소 유니크 elapsed_time 개수: {min_count} (hadm_id: {min_hadm_id})")

# 추가 통계 정보
print(f"전체 hadm_id 수: {unique_elapsed_time_counts.count()}")
print(f"유니크 elapsed_time 개수의 평균: {unique_elapsed_time_counts.mean():.2f}")
print(f"유니크 elapsed_time 개수의 중앙값: {unique_elapsed_time_counts.median():.2f}")

# 결과를 DataFrame으로 저장 (옵션)
output_file = "./train_F/hadm_id_elapsed_time_counts.csv"
unique_elapsed_time_counts.to_frame("unique_elapsed_time_count").reset_index().to_csv(output_file, index=False)
print(f"hadm_id별 유니크 elapsed_time 개수를 저장한 파일 경로: {output_file}")


In [ ]:
import pandas as pd

# 파일 경로 정의
input_file = "./train_F/timeline2.csv"
dictionary_file = "./dictionary/d_items.csv"
output_file = "./train_F/top_itemids_sorted_by_diff.csv"

# 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(input_file)

# d_items.csv 로드
print("dictionary/d_items.csv 데이터 로드 중...")
d_items = pd.read_csv(dictionary_file)

# d_items에서 itemid, label, category만 추출
if "itemid" not in d_items.columns or "label" not in d_items.columns or "category" not in d_items.columns:
    raise KeyError("d_items.csv에 'itemid', 'label', 'category' 열이 필요합니다.")
d_items = d_items[["itemid", "label", "category"]]

# timeline2.csv에 label과 category 병합
print("label, category 병합 중...")
df = df.merge(d_items, on="itemid", how="left")

# event_type별로 상위 30개 itemid 추출
print("event_type별로 상위 30개 itemid, label, category 추출 중...")
top_itemids_per_event_type = (
    df.groupby("event_type")["itemid"]
    .value_counts()
    .groupby(level=0)
    .head(30)  # 상위 30개 추출
    .reset_index(name="count")
)

# itemid에 해당하는 label, category를 병합
print("label, category 추가 중...")
result = top_itemids_per_event_type.merge(df[["itemid", "label", "category"]].drop_duplicates(), on="itemid", how="left")

# 생존자 및 사망자 비율 계산
print("생존자와 사망자 비율 계산 중...")
total_survivors = df[df["hospital_expire_flag"] == 0]["hadm_id"].nunique()
total_deaths = df[df["hospital_expire_flag"] == 1]["hadm_id"].nunique()

# 각 itemid별 생존자와 사망자 중 사용된 비율 계산
usage_stats = df.groupby(["itemid", "hospital_expire_flag"])["hadm_id"].nunique().unstack(fill_value=0)
usage_stats["survivor_ratio"] = usage_stats[0] / total_survivors  # 생존자 비율
usage_stats["death_ratio"] = usage_stats[1] / total_deaths        # 사망자 비율
usage_stats = usage_stats.reset_index()

# result에 생존자와 사망자 비율 추가
result = result.merge(usage_stats[["itemid", "survivor_ratio", "death_ratio"]], on="itemid", how="left")

# survivor_ratio와 death_ratio 차이의 절댓값 추가
print("survivor_ratio와 death_ratio 차의 절댓값 계산 중...")
result["ratio_diff_abs"] = (result["survivor_ratio"] - result["death_ratio"]).abs()

# ratio_diff_abs 기준으로 내림차순 정렬
sorted_result = result.sort_values("ratio_diff_abs", ascending=False)

# 결과 저장
print("결과 저장 중...")
sorted_result.to_csv(output_file, index=False)
print(f"결과 저장 완료: {output_file}")

# pandas 출력 옵션 설정 (줄바꿈 방지 및 넓은 열 표시)
pd.set_option("display.max_colwidth", None)  # 줄바꿈 없이 전체 열 표시
pd.set_option("display.expand_frame_repr", False)  # 열이 좁아서 줄바꿈되는 것 방지

# 결과 출력
print("결과:")
print(sorted_result.to_string(index=False))  # index=False로 출력에서 인덱스 제거


In [ ]:
import pandas as pd

# 파일 경로 정의
input_file = "./train_F/timeline2.csv"
dictionary_file = "./dictionary/d_items.csv"
top_items_file = "./train_F/top_itemids_sorted_by_diff.csv"
output_file = "./train_F/timeline3.csv"

# 1. 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(input_file)
print("dictionary/d_items.csv 데이터 로드 중...")
d_items = pd.read_csv(dictionary_file)
print("top_itemids_sorted_by_diff.csv 데이터 로드 중...")
top_items = pd.read_csv(top_items_file)

# 2. d_items에서 필요한 열 추출
if "itemid" not in d_items.columns or "label" not in d_items.columns:
    raise KeyError("d_items.csv에 'itemid'와 'label' 열이 필요합니다.")
d_items = d_items[["itemid", "label"]]

# 3. top_itemids_sorted_by_diff에서 중요한 itemid 추출
if "itemid" not in top_items.columns:
    raise KeyError("top_itemids_sorted_by_diff.csv에 'itemid' 열이 필요합니다.")
important_itemids = top_items["itemid"].unique()

# 4. hadm_id별 데이터 필터링
print("hadm_id별 데이터 필터링 중...")
filtered_dfs = []
for hadm_id, group in df.groupby("hadm_id"):
    if len(group) > 200:
        # 중요 itemid만 필터링
        filtered_group = group[group["itemid"].isin(important_itemids)]
        filtered_dfs.append(filtered_group)
    else:
        filtered_dfs.append(group)  # 200개 이하인 데이터는 그대로 유지

# 필터링된 데이터 결합
df_filtered = pd.concat(filtered_dfs)

# 5. itemid를 label로 변경
print("itemid를 label로 변경 중...")
df_filtered = df_filtered.merge(d_items, on="itemid", how="left")

# 매칭되지 않은 itemid 확인
unmatched = df_filtered[df_filtered["label"].isna()]
unmatched_count = len(unmatched)
if unmatched_count > 0:
    print(f"매칭되지 않은 itemid 개수: {unmatched_count}")
    print(f"매칭되지 않은 itemid 목록:\n{unmatched['itemid'].unique()}")

# 매칭되지 않은 itemid를 "Unknown Item"으로 대체
df_filtered["label"] = df_filtered["label"].fillna("Unknown Item")

# itemid 열을 label로 대체
df_filtered["itemid"] = df_filtered["label"]
df_filtered = df_filtered.drop(columns=["label"])

# 6. 결과 저장
print("결과 저장 중...")
df_filtered.to_csv(output_file, index=False)
print(f"필터링 및 itemid -> label 변경된 파일 저장 완료: {output_file}")


In [ ]:
import pandas as pd

# 결과 파일 경로
output_file = "./train_F/timeline3.csv"

# 파일 로드
print("결과 파일 로드 중...")
df = pd.read_csv(output_file)

# 고유 hadm_id 개수 확인
unique_hadm_ids = df["hadm_id"].nunique()
print(f"고유 hadm_id 개수: {unique_hadm_ids}")


In [ ]:
import pandas as pd
import json

# 파일 경로 정의
input_file = "./train_F/timeline3.csv"
bert_input_file = "./train_F/bert_input.json"

# 최대 길이 설정
MAX_FEATURE_LENGTH = 512

# 데이터 로드
print("데이터 로드 중...")
df = pd.read_csv(input_file)

# 문자열 결합 함수 정의
def create_combined_word(event_type, itemid):
    # 공백을 `_`로 대체
    event_type_clean = str(event_type).replace(" ", "_")
    itemid_clean = str(itemid).replace(" ", "_")
    return f"{event_type_clean}_{itemid_clean}"

# event_type과 itemid를 결합하여 새로운 열 생성
print("event_type과 itemid를 결합 중...")
df["combined_word"] = df.apply(lambda row: create_combined_word(row["event_type"], row["itemid"]), axis=1)

# hadm_id별 데이터 정리
print("hadm_id별 데이터 정리 중...")
hadm_id_grouped = df.groupby("hadm_id")

# BERT 입력 데이터 생성
print("BERT 입력 데이터 생성 중...")
bert_inputs = []
for hadm_id, group in hadm_id_grouped:
    # 결합된 단어 리스트 생성 (최대 길이 제한)
    combined_words = group["combined_word"].tolist()[:MAX_FEATURE_LENGTH]
    hospital_expire_flag = int(group["hospital_expire_flag"].iloc[0])  # 정답 레이블 (같은 hadm_id는 동일한 flag)

    # 한 문장으로 결합
    sentence = " ".join(combined_words)

    # BERT 입력 형식으로 데이터 저장
    bert_inputs.append({
        "hadm_id": int(hadm_id),
        "sentence": sentence,
        "label": hospital_expire_flag
    })

# 결과 저장 (JSON 파일로 저장)
print("결과 저장 중...")
with open(bert_input_file, "w") as f:
    json.dump(bert_inputs, f, indent=4)

print(f"BERT 입력 데이터 생성 완료. 저장된 파일 경로: {bert_input_file}")
